In [ ]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

tags_link=[]
all_articles = []


### get tags on dev.to

In [5]:
firefox_option = Options()
firefox_option.add_argument("--headless")
firefox_option.add_argument("--disable-blink-features=AutomationControlled")
firefox_option.add_argument(f"user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
driver = webdriver.Firefox(options=firefox_option)
driver.implicitly_wait(10)
driver.get("https://dev.to/")
soup = BeautifulSoup(driver.page_source, "html.parser")
articles = soup.select('.sidebar-nav-element')
tags=[]
for article in articles:
    try:
        link = article.find('a', href=True)     
        tags.append(link['href'])
    except Exception as e:
        print(f"Erreur sur un article: {e}")

driver.quit()

In [ ]:
# dev.to scraping
def scrape_tag(tag):
    """Scrape a single tag with fresh driver instance"""
    firefox_option = Options()
    firefox_option.add_argument("--headless")
    firefox_option.add_argument("--disable-blink-features=AutomationControlled")
    firefox_option.add_argument(f"user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    

    driver = webdriver.Firefox(options=firefox_option)
    articles_data = []
    firefox_option = Options()
    firefox_option.add_argument("--headless")
    firefox_option.add_argument("--disable-blink-features=AutomationControlled")
    firefox_option.add_argument(f"user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    

    driver = webdriver.Firefox(options=firefox_option)
    articles_data = []
    
    try:
        print(f"Scraping tag: {tag} {tags.index(tag)+1}/{len(tags)}")
        url = f"https://dev.to{tag}"
        driver.get(url)
        
        # Wait for articles
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "main"))
        )
        time.sleep(2)
        
        # Scroll
       
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 10);")
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        articles = soup.select('.crayons-story')
        print(f"Trouvé {len(articles)} articles pour {tag}")
        
        for article in articles:
            try:
                link = article.find('a', href=True)     
                articles_data.append(link['href'])
            except Exception as e:
                print(f"Erreur sur un article: {e}")
                continue
                
    except Exception as e:
        print(f"Erreur pour {tag}: {e}")
    
    finally:
        driver.quit()
    
    return articles_data


for tag in tags:
    links = scrape_tag(tag)
    tags_link.extend(links)
    time.sleep(3)  # Delay between tags

print(tags_link)


Scraping tag: /t/webdev 1/30
Trouvé 25 articles pour /t/webdev
Scraping tag: /t/programming 2/30
Trouvé 25 articles pour /t/programming
Scraping tag: /t/ai 3/30
Trouvé 25 articles pour /t/ai
Scraping tag: /t/javascript 4/30
Trouvé 25 articles pour /t/javascript
Scraping tag: /t/beginners 5/30
Trouvé 25 articles pour /t/beginners
Scraping tag: /t/tutorial 6/30
Trouvé 25 articles pour /t/tutorial
Scraping tag: /t/productivity 7/30
Trouvé 25 articles pour /t/productivity
Scraping tag: /t/python 8/30
Trouvé 25 articles pour /t/python
Scraping tag: /t/devops 9/30
Trouvé 25 articles pour /t/devops
Scraping tag: /t/react 10/30
Trouvé 25 articles pour /t/react
Scraping tag: /t/opensource 11/30
Trouvé 25 articles pour /t/opensource
Scraping tag: /t/career 12/30
Trouvé 25 articles pour /t/career
Scraping tag: /t/discuss 13/30
Trouvé 25 articles pour /t/discuss
Scraping tag: /t/security 14/30
Trouvé 25 articles pour /t/security
Scraping tag: /t/aws 15/30
Trouvé 25 articles pour /t/aws
Scraping ta

### save it for re use it

In [ ]:
df = pd.Series(tags_link, name="article_links")
df.to_csv("../data/raw/devto_article_links.csv", index=False)


In [4]:
tags_links = pd.read_csv("../data/raw/devto_article_links.csv")["article_links"].tolist()
tags_links

['https://dev.to/dannwaneri/were-creating-a-knowledge-collapse-and-no-ones-talking-about-it-226d',
 'https://dev.to/rachynska/if-its-not-accessible-its-not-design-1l5p',
 'https://dev.to/polliog/building-real-time-log-streaming-with-postgresql-listennotify-4cbj',
 'https://dev.to/ingosteinke/website-builders-vs-web-designers-and-developers-2ef7',
 'https://dev.to/magnificode/building-opentrainer-real-time-workout-tracking-with-convex-and-nextjs-59h4',
 'https://dev.to/iamovi/i-made-buttons-that-run-away-from-you-prank-projects-5ej0',
 'https://dev.to/asamaes/freelance-rate-negotiation-coach-ai-powered-salary-intelligence-3kn2',
 'https://dev.to/fern_d3v/burnout-in-learning-4fb6',
 'https://dev.to/dromi_monster_56baf07d6f0/understanding-network-devices-2o6o',
 'https://dev.to/thebackenddev/pgp-and-gpg-explained-encrypt-files-and-verify-authenticity-bko',
 'https://dev.to/fabianosalles/why-most-svg-to-png-converters-are-overkill-and-what-i-actually-needed-2j52',
 'https://dev.to/brennan/

In [ ]:
# ===============================
# DEV.TO ARTICLE SCRAPER (SAFE)
# ONE CELL – APPEND TO CSV
# ===============================

OUT_PATH = "../data/raw/dev-to.csv"

def dev_to_scraper(url: str) -> dict:
    firefox_option = Options()
    firefox_option.add_argument("--headless")
    driver = webdriver.Firefox(options=firefox_option)

    article_data = {
        "url": url,
        "source": "dev.to",
        "title": None,
        "author": None,
        "published_at": None,
        "content": None,
        "tags": None,
        "error": None,
    }

    try:
        driver.set_page_load_timeout(25)
        driver.get(url)

        WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.TAG_NAME, "main"))
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")

        title_el = soup.select_one(".crayons-article__header__meta h1")
        body = soup.select_one(".crayons-article__body")

        if not title_el or not body:
            raise ValueError("Article body or title not found")

        article_data["title"] = title_el.get_text(strip=True)

        author_el = soup.select_one(".crayons-article__header__meta a")
        article_data["author"] = author_el.get_text(strip=True) if author_el else None

        time_el = soup.select_one("time[datetime]")
        article_data["published_at"] = time_el["datetime"] if time_el else None

        parts = []
        for el in body.find_all(["h1", "h2", "h3", "p", "li"], recursive=True):
            text = el.get_text(" ", strip=True)
            if not text:
                continue
            if el.name in ("h1", "h2", "h3"):
                parts.append(f"\n## {text}\n")
            elif el.name == "li":
                parts.append(f"- {text}")
            else:
                parts.append(text)

        article_data["content"] = "\n".join(parts).strip()

        tags = [t.get_text(strip=True) for t in soup.select(".crayons-tag")]
        article_data["tags"] = ",".join(tags)

    except Exception as e:
        article_data["error"] = str(e)

    finally:
        driver.quit()

    return article_data


# ===============================
# MAIN LOOP (APPEND PER ARTICLE)
# ===============================

write_header = not os.path.exists(OUT_PATH)

for i, url in enumerate(tags_links, start=1):
    article = dev_to_scraper(url)

    title = article["title"] if article["title"] else "(no title)"
    status = "OK" if not article["error"] else f"FAIL: {article['error']}"
    print(f"[{i}/{len(tags_links)}] {status} | {title}")

    pd.DataFrame([article]).to_csv(
        OUT_PATH,
        mode="a",
        header=write_header,
        index=False
    )
    write_header = False

    time.sleep(1)


[1/750] OK | We're Creating a Knowledge Collapse and No One's Talking About It
[2/750] OK | If It's Not Accessible, It's Not Design
[3/750] OK | Building Real-Time Log Streaming with PostgreSQL LISTEN/NOTIFY
[4/750] OK | Website Builders vs. Web Designers and Developers
[5/750] OK | I Didn't Like Other Workout Tracking Apps, So I Built My Own.
[6/750] OK | i made buttons that run away from you (prank projects)
[7/750] OK | 💼 Freelance Rate Negotiation Coach - AI-Powered Salary Intelligence
[8/750] OK | burnout in learning
[9/750] OK | Understanding Network Devices
[10/750] OK | PGP and GPG Explained: Encrypt Files and Verify Authenticity
[11/750] OK | Why most SVG to PNG converters are overkill (and what I actually needed)
[12/750] OK | What I Have Learned Being on the IndieWeb for a Month
[13/750] OK | I stopped analysing matches and started building a tool to decide whether to go
[14/750] OK | Building an MCP Gateway: Lessons from Production
[15/750] OK | How to Migrate from Clerk to

In [ ]:
# dev.to scraping
def scrape_tag(tag):
    """Scrape a single tag with fresh driver instance"""
    firefox_option = Options()
    firefox_option.add_argument("--headless")
    firefox_option.add_argument("--disable-blink-features=AutomationControlled")
    firefox_option.add_argument(f"user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    

    driver = webdriver.Firefox(options=firefox_option)
    articles_data = []
    firefox_option = Options()
    firefox_option.add_argument("--headless")
    firefox_option.add_argument("--disable-blink-features=AutomationControlled")
    firefox_option.add_argument(f"user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    

    driver = webdriver.Firefox(options=firefox_option)
    articles_data = []
    
    try:
        print(f"Scraping tag: {tag} {tags.index(tag)+1}/{len(tags)}")
        url = f"https://dev.to{tag}"
        driver.get(url)
        
        # Wait for articles
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "main"))
        )
        time.sleep(2)
        
        # Scroll
       
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 10);")
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        articles = soup.select('.crayons-story')
        print(f"Trouvé {len(articles)} articles pour {tag}")
        
        for article in articles:
            try:
                link = article.find('a', href=True)     
                articles_data.append(link['href'])
            except Exception as e:
                print(f"Erreur sur un article: {e}")
                continue
                
    except Exception as e:
        print(f"Erreur pour {tag}: {e}")
    
    finally:
        driver.quit()
    
    return articles_data


for tag in tags:
    links = scrape_tag(tag)
    tags_link.extend(links)
    time.sleep(3)  # Delay between tags

print(tags_link)


In [21]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

def github_resources_scrape(topic: str, max_pages: int = 50):
    firefox_options = Options()
    firefox_options.add_argument("--headless")

    driver = webdriver.Firefox(options=firefox_options)

    all_links = []
    seen = set()

    try:
        print(f"Scraping topic: {topic}")

        for page in range(1, max_pages + 1):
            url = f"https://github.com/resources/articles?topic={topic}&page={page}"
            driver.get(url)

            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "main"))
            )

            soup = BeautifulSoup(driver.page_source, "html.parser")

            new_links = []
            for a in soup.select('a[href^="https://github.com/resources/articles/"]'):
                href = a.get("href")
                if not href:
                    continue
                

                if href not in seen:
                    seen.add(href)
                    new_links.append(href)

            if not new_links:
                print(f"No new links on page {page} → stop for topic={topic}")
                break

            print(f"Page {page}: +{len(new_links)} links")
            all_links.extend(new_links)

            time.sleep(1)

    finally:
        driver.quit()

    return all_links


topics = ["ai", "software-development", "devops", "security"]
github_links = []

for t in topics:
    github_links.extend(github_resources_scrape(t))
    time.sleep(2)

print("Total:", len(github_links))
print(github_links[:10])


Scraping topic: ai
Page 1: +12 links
Page 2: +5 links
No new links on page 3 → stop for topic=ai
Scraping topic: software-development
Page 1: +12 links
Page 2: +11 links
Page 3: +1 links
No new links on page 4 → stop for topic=software-development
Scraping topic: devops
Page 1: +12 links
Page 2: +5 links
No new links on page 3 → stop for topic=devops
Scraping topic: security
Page 1: +12 links
Page 2: +11 links
No new links on page 3 → stop for topic=security
Total: 81
['https://github.com/resources/articles/software-development-with-retrieval-augmentation-generation-rag', 'https://github.com/resources/articles/what-is-prompt-engineering', 'https://github.com/resources/articles/what-is-aiops', 'https://github.com/resources/articles/what-is-generative-ai-genai', 'https://github.com/resources/articles/what-are-neural-networks', 'https://github.com/resources/articles/what-is-open-source-ai', 'https://github.com/resources/articles/what-is-vibe-coding', 'https://github.com/resources/articles

In [23]:
len(github_links)

81

In [ ]:
# ===============================
# GITHUB RESOURCES ARTICLE SCRAPER (SAFE)
# ONE CELL – APPEND TO CSV
# ===============================

from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

OUT_PATH = "../data/raw/github-resources.csv"

def github_resources_article_scraper(url: str) -> dict:
    firefox_options = Options()
    firefox_options.add_argument("--headless")
    driver = webdriver.Firefox(options=firefox_options)

    article_data = {
        "url": url,
        "source": "github_resources",
        "title": None,
        "author": "GitHub",        
        "published_at": None,
        "content": None,
        "tags": None,
        "error": None,
    }

    try:
        driver.set_page_load_timeout(25)
        driver.get(url)

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "main"))
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # -------- TITLE --------
        title_el = soup.select_one("main h1") or soup.select_one("h1")
        if not title_el:
            raise ValueError("Title not found")
        article_data["title"] = title_el.get_text(" ", strip=True)

        # -------- DATE --------
        time_el = soup.select_one("time[datetime]")
        article_data["published_at"] = time_el["datetime"] if time_el else None

        # -------- CONTENT (Prose container) --------
        container = (
            soup.select_one("main div[class*='Prose']")
            or soup.select_one("article div[class*='Prose']")
        )

        if not container:
            raise ValueError("Prose content container not found")

        parts = []
        for el in container.find_all(["h1", "h2", "h3", "p", "li"], recursive=True):
            text = el.get_text(" ", strip=True)
            if not text:
                continue

            if el.name in ("h1", "h2", "h3"):
                parts.append(f"\n## {text}\n")
            elif el.name == "li":
                parts.append(f"- {text}")
            else:
                parts.append(text)

        content = "\n".join(parts).strip()

        if len(content) < 200:
            raise ValueError("Content too short (blocked or invalid page)")

        article_data["content"] = content

        # -------- TAGS (best effort) --------
        tag_elems = soup.select(
            "a[href*='?topic='], a[href*='/resources/articles?topic=']"
        )
        tags = []
        seen = set()

        for a in tag_elems:
            t = a.get_text(" ", strip=True)
            if t and t.lower() not in ("articles", "resources") and t not in seen:
                seen.add(t)
                tags.append(t)

        article_data["tags"] = ",".join(tags) if tags else None

    except Exception as e:
        article_data["error"] = str(e)

    finally:
        driver.quit()

    return article_data


# ===============================
# MAIN LOOP (APPEND PER ARTICLE)
# ===============================

write_header = not os.path.exists(OUT_PATH)

for i, url in enumerate(github_links, start=1):
    article = github_resources_article_scraper(url)

    title = article["title"] if article["title"] else "(no title)"
    status = "OK" if not article["error"] else f"FAIL: {article['error']}"
    print(f"[{i}/{len(github_links)}] {status} | {title}")

    pd.DataFrame([article]).to_csv(
        OUT_PATH,
        mode="a",
        header=write_header,
        index=False
    )
    write_header = False

    time.sleep(1)

print("Done. Saved to:", OUT_PATH)


[1/81] OK | What is retrieval-augmented generation (RAG)?
[2/81] OK | What is prompt engineering?
[3/81] OK | What is AIOps?
[4/81] OK | What is generative AI (GenAI)?
[5/81] OK | What are neural networks?
[6/81] OK | What is open source AI?
[7/81] OK | What Is Vibe Coding?
[8/81] OK | What is Agentic AI?
[9/81] OK | What are Generative AI Models?
[10/81] OK | AI in Software Development
[11/81] OK | AI Code Reviews
[12/81] OK | Machine Learning (ML) in Software Development
[13/81] OK | What are AI models?
[14/81] OK | What are AI agents?
[15/81] OK | What is Natural language processing (NLP)?
[16/81] OK | What is AI code generation?
[17/81] OK | AI coding tools for beginner and expert coders
[18/81] OK | What is prompt engineering?
[19/81] OK | What is an integrated development environment (IDE)?
[20/81] OK | What is open source AI?
[21/81] OK | What is a CLI (command-line interface)?
[22/81] OK | What is an API?
[23/81] OK | What is an SDK?
[24/81] OK | What is the SDLC?
[25/81] OK | 

Problem reading geckodriver versions: error sending request for url (https://raw.githubusercontent.com/SeleniumHQ/selenium/trunk/common/geckodriver/geckodriver-support.json). Using latest geckodriver version


[56/81] OK | What is DevSecOps?
[57/81] OK | DevOps monitoring tools: Automating your DevOps monitoring processes
[58/81] OK | What is the DevOps Model? Exploring foundational practices in DevOps
[59/81] OK | What is AIOps?
[60/81] OK | What is application modernization?
[61/81] OK | What is a software bill of materials (SBOM)?
[62/81] OK | What is a Data Breach?
[63/81] OK | What is Code Scanning?
[64/81] OK | What is Cross-Site Scripting (XSS)
[65/81] OK | What is risk-based vulnerability management (RBVM)?
[66/81] OK | What is secret scanning?
[67/81] OK | What is a security risk assessment?
[68/81] OK | What is software supply chain security?
[69/81] OK | What Is Incident Response?
[70/81] OK | What is threat modeling?
[71/81] OK | What is runtime application self-protection (RASP)?
[72/81] OK | What is static application security testing (SAST)?
[73/81] OK | What is shift left?
[74/81] FAIL: Message: Reached error page: about:neterror?e=dnsNotFound&u=https%3A//github.com/resources

In [3]:
# ===============================
# MERGE DEV.TO + GITHUB DATASETS
# ONE CELL
# ===============================

import pandas as pd
import os

DEVTO_PATH = "../data/raw/dev-to.csv"
GITHUB_PATH = "../data/raw/github-resources.csv"
TECHCRUNCH_PATH = "../data/raw/tech_watch_raw_data.csv"
OUT_PATH = "../data/raw/articles_combined.csv"

# Load datasets
df_devto = pd.read_csv(DEVTO_PATH)
df_github = pd.read_csv(GITHUB_PATH)
df_techcrunch = pd.read_csv(TECHCRUNCH_PATH)

print("Dev.to articles:", len(df_devto))
print("GitHub articles:", len(df_github))
print("TechCrunch articles:", len(df_techcrunch))
# Combine
df_all = pd.concat([df_devto, df_github , df_techcrunch], ignore_index=True)

# Optional but recommended: remove duplicates by URL
df = df_all.drop_duplicates(subset=["url"])


print(f"\nTotal articles scrapés: {len(df)}")
print(df.head())
df.to_csv('../data/raw/articles.csv', index=False)
# Optional: remove rows with no content (failed scrapes)
df_all = df_all[df_all["content"].notna()]

print("Combined articles:", len(df_all))

# Save
df_all.to_csv(OUT_PATH, index=False)

print("Saved combined dataset to:", OUT_PATH)


Dev.to articles: 750
GitHub articles: 81
TechCrunch articles: 574

Total articles scrapés: 1158
                                                 url  source  \
0  https://dev.to/dannwaneri/were-creating-a-know...  dev.to   
1  https://dev.to/rachynska/if-its-not-accessible...  dev.to   
2  https://dev.to/polliog/building-real-time-log-...  dev.to   
3  https://dev.to/ingosteinke/website-builders-vs...  dev.to   
4  https://dev.to/magnificode/building-opentraine...  dev.to   

                                               title author  \
0  We're Creating a Knowledge Collapse and No One...    NaN   
1            If It's Not Accessible, It's Not Design    NaN   
2  Building Real-Time Log Streaming with PostgreS...    NaN   
3  Website Builders vs. Web Designers and Developers    NaN   
4  I Didn't Like Other Workout Tracking Apps, So ...    NaN   

           published_at                                            content  \
0  2026-01-27T01:59:58Z  "Hostile experts created the dataset 